# Wavelet analysis for June experiment

In [25]:
import polars as pl
from pathlib import Path
import numpy as np
import pyboat
from pyboat import WAnalyzer
from pyboat import plotting
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import pandas as pd
from scipy.signal import savgol_filter
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import matplotlib.colors as mcolors
from natsort import natsorted

from dataclasses import dataclass
from typing import Generator

@dataclass
class WaveResults:
    """Wavlet analysis results"""
    ridge_df: pl.DataFrame
    summary_df: pl.DataFrame
    periods: np.ndarray
    fourier_df: pl.DataFrame

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

def savgol_smooth(
    dataset: pl.DataFrame, 
    value_col: str,
    well_col: str,
    window_length: int = 13, 
    polyorder: int = 3
    ) -> pl.DataFrame:
    return (
        dataset
        .group_by(well_col)
        .map_groups(
            lambda g: 
                g.with_columns(
                    pl.Series(value_col, savgol_filter(
                        g[value_col].to_numpy(),
                        window_length=window_length,
                        polyorder=polyorder
                    ))
                )
        )
    )
    
def calculate_wavelets(
    dataset: pl.DataFrame,
    name_col: str, #this is like the group_label but the actual name of it
    well_list: list[str],
    well_col: str,
    stat_val: str,
    timepoint_col: str,
    sampling_interval: float = 1.0,
    period_range: tuple[int, int] = (18,40),
    n_periods: int=200,
    ridge_thresh: int=0,
) -> WaveResults:
    periods = np.linspace(period_range[0], period_range[1], n_periods)
    ridge_results = []
    fourier_results = []
    summary_rows = []
    
    for well in tqdm(well_list, desc='Wells'):
        well_df = (dataset
                   .filter(pl.col(well_col) == well)
                   .sort(timepoint_col)
                   )
        signal = well_df[stat_val].to_numpy()
        tpoints = well_df[timepoint_col].to_numpy()
        group = well_df[name_col].unique().item()
        
        # Wave analyzer
        wan = WAnalyzer(periods=periods, dt=sampling_interval, time_unit_label='h')
        
        # compute spectrum
        wan.compute_spectrum(signal, do_plot=False)
        
        ridge = wan.get_maxRidge(power_thresh=ridge_thresh)
        fourier_power = wan.get_averaged_spectrum()
        
        # ridge summaries
        ridge['time'] = tpoints
        ridge[well_col] = well
        ridge[name_col] = group
        ridge_results.extend(ridge.to_dict(orient='records'))
        
        for p, pw in zip(periods, fourier_power):
            fourier_results.append({
                well_col: well,
                name_col: group,
                'period': p,
                'power': pw
            })
        
        # summaries
        mid_idx = len(ridge) // 2
        summary_rows.append({
            well_col          : well,
            name_col          : group,
            'mean_period'     : ridge['periods'].mean(),
            'std_period'      : ridge['periods'].std(),
            'median_period'   : ridge['periods'].median(),
            'mean_amplitude'  : ridge['amplitude'].mean(),
            'std_amplitude'   : ridge['amplitude'].std(),
            'median_amplitude': ridge['amplitude'].median(),
            'mid_phase'       : ridge['phase'].iloc[mid_idx],
            'peak_fourier_period': periods[np.argmax(fourier_power)],
        })
    
    return WaveResults(
        ridge_df=(pl.DataFrame(ridge_results)
                  .with_columns(
                      (pl.col('phase') / (2 * np.pi) * pl.col("periods"))
                      .alias("phase_hours")
                      )),
        fourier_df=pl.DataFrame(fourier_results),
        periods=periods,
        summary_df=pl.DataFrame(summary_rows)
    )    
  
def plot_wavelets(
    wavelet_results: WaveResults,
    dataset: pl.DataFrame,
    well_list: list[str],
    well_col: str,
    name_col: str,
    stat_val: str,
    timepoint_col: str,
    win_len: int = 13,
    polyorder: int = 3,
    tick_intervals: int = 12
) -> Generator[plt.Figure]:
    
    ridge_df = wavelet_results.ridge_df.to_pandas()
    fourier_df = wavelet_results.fourier_df.to_pandas()
    periods = wavelet_results.periods
    
    for well in tqdm(well_list, desc='well plots'):
        well_signal = (
            dataset
            .filter(pl.col(well_col) == well)
            .sort(timepoint_col)
            .select([timepoint_col, stat_val])
            .to_pandas()
        )
        tpoints = well_signal[timepoint_col].to_numpy()
        signal = well_signal[stat_val].to_numpy()
        
        smoothed = savgol_filter(signal, window_length=win_len, polyorder=polyorder)
        well_signal[stat_val] = smoothed
        
        well_ridge = ridge_df[ridge_df[well_col] == well]
        well_fourier = fourier_df[fourier_df[well_col] == well]
        group = well_ridge[name_col].iloc[0]
        
        # plots
        fig, axs = plt.subplots(3, 1, figsize=(14,12))
        
        # detrended signal plot
        
        sns.lineplot(data=well_signal, 
                     x=timepoint_col, 
                     y=stat_val,
                     color='royalblue', 
                     linewidth=1.5,
                     ax=axs[0])
        axs[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
        axs[0].set_xlabel('Time (hours)')
        axs[0].set_ylabel('Expression')
        axs[0].set_title('Detrended Signal')
        axs[0].set_xlim(tpoints[0], tpoints[-1])
        add_period_marks(axs[0], tpoints[0], tpoints[-1], tick_intervals)
        
        # wavelet spectrum
        
        wan = WAnalyzer(periods=periods, 
                        dt=(tpoints[1] - tpoints[0]), 
                        time_unit_label='h')
        wan.compute_spectrum(signal, do_plot=False)
        extent = [tpoints[0], tpoints[-1], periods[-1], periods[0]]
        axs[1].imshow(np.abs(wan.transform), aspect='auto', extent=extent, 
                      cmap='viridis', origin='upper')
        axs[1].plot(well_ridge['time'], well_ridge['periods'],
                       color='cyan', linewidth=2, label='ridge')
        axs[1].set_xlabel('Time (hours)')
        axs[1].set_ylabel('Period (hours)')
        axs[1].set_title('Wavelet Power Spectrum')
        axs[1].legend(fontsize=8)
        axs[1].set_xlim(tpoints[0], tpoints[-1])  # ← force x limits explicitly
        axs[1].set_ylim(periods[-1], periods[0])  # ← force y limits explicitly
        add_period_marks(axs[1], tpoints[0], tpoints[-1], tick_intervals)
        
        # fourier spectrum
        peak_period = well_fourier.loc[well_fourier['power'].idxmax(), 'period']
        axs[2].plot(well_fourier['period'], well_fourier['power'],
                    color='darkred', linewidth=1.5)
        axs[2].axvline(peak_period, color='red', linewidth=1.5, linestyle='--',
                        label=f'Peak: {peak_period:.1f}h')
        axs[2].set_xlabel('Period (hours)')
        axs[2].set_ylabel('Power')
        axs[2].set_title('Fourier Spectrum (time-averaged)')
        axs[2].legend(fontsize=8)
        axs[2].set_xlim(periods[0], periods[-1])
        # add_period_marks(axs[2], periods[0], periods[-1])
        
        fig.suptitle(f"Well: {well} - {group}", fontsize=11)
        fig.tight_layout()
        
        yield fig
        
def interactive_ridges(
    wavelet_res: WaveResults,
    name_col: str,
    metrics: list[str],
    out_file: Path
    ) -> None:
    
    timepoint_col = 'time'
    ridge_df = wavelet_res.ridge_df
    
    replicate_agg = (
        ridge_df
        .group_by([name_col, timepoint_col])
        .agg([
            *[pl.col(m).mean().alias(f'{m}_mean') for m in metrics],
            *[pl.col(m).std().alias(f'{m}_std') for m in metrics]
        ])
        .sort([name_col, timepoint_col])
        .to_pandas()
    )
    
    groups = sorted(replicate_agg[name_col].unique())
    palette = px.colors.qualitative.Plotly
    colormap = {g:palette[i % len(palette)] for i, g in enumerate(groups)}
    
    fig = make_subplots(
        rows=len(metrics), cols=1,
        shared_xaxes=False,
        subplot_titles=[m.capitalize() for m in metrics],
        vertical_spacing=0.05
    )
    
    for i, group in enumerate(groups):
        color = colormap[group]
        color_rgba = f'rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}'
        
        gdf = replicate_agg[replicate_agg[name_col] == group]
        
        for row_idx, metric in enumerate(metrics, start=1):
            mean_col = f'{metric}_mean'
            std_col = f'{metric}_std'
            
            show_legend = (row_idx == 1)
            
            # shading
            fig.add_trace(go.Scatter(
                x=pd.concat([gdf[timepoint_col], gdf[timepoint_col].iloc[::-1]]),
                y=pd.concat([
                    gdf[mean_col] + gdf[std_col],
                    (gdf[mean_col] - gdf[std_col]).iloc[::-1]
                ]),
                fill='toself',
                fillcolor=f'{color_rgba}, 0.15)',
                line=dict(color='rgba(255,255,255,0)'),
                showlegend=False,
                legendgroup=group,
                name=group,
                hoverinfo='skip',
            ), row=row_idx, col=1)
            
            # mean line
            fig.add_trace(go.Scatter(
                x=gdf[timepoint_col],
                y=gdf[mean_col],
                mode='lines',
                name=group,
                legendgroup=group,
                showlegend=show_legend,
                line=dict(color=color, width=2),
                hovertemplate=f'<b>{group}</b><br>Time: %{{x:.1f}}h<br>{metric}: %{{y:.3f}}<extra></extra>',
            ), row=row_idx, col=1)
    
    fig.update_layout(
        # height=300 * len(metrics),
        height=2000,
        title='Wavelet Ridge Analysis — All Groups',
        hovermode='x unified',
        legend=dict(
            groupclick='toggleitem',   # click legend to show/hide individual groups
            x=1.02,
            y=1,
        ),
    )

    fig.update_xaxes(title_text='Time (hours)', row=len(metrics), col=1)
    for row_idx, metric in enumerate(metrics, start=1):
        fig.update_yaxes(title_text=metric.capitalize(), row=row_idx, col=1)


    fig.update_traces(visible='legendonly')
    fig.write_html(out_file,
        include_plotlyjs=True,
        post_script="""
        // Override legend click to toggle ALL traces with same legendgroup
        var myPlot = document.getElementsByClassName('plotly-graph-div')[0];
        myPlot.on('plotly_legendclick', function(data) {
            var clickedGroup = data.data[data.curveNumber].legendgroup;
            var update = {};
            var indices = [];
            
            // find all traces with same legendgroup
            data.data.forEach(function(trace, i) {
                if (trace.legendgroup === clickedGroup) {
                    indices.push(i);
                }
            });
            
            // check current visibility of clicked trace
            var currentVis = data.data[data.curveNumber].visible;
            var newVis = (currentVis === 'legendonly' || currentVis === false) ? true : 'legendonly';
            
            var visUpdate = indices.map(function() { return newVis; });
            Plotly.restyle(myPlot, {'visible': visUpdate}, indices);
            
            return false; // prevent default behavior
        });
        """)
    fig.show()

*Note*: The platemap processing was done in `06-01_envision_visualization.ipynb`

In [27]:
detrended_dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/enVision/enVision_preprocessed/2026-06-01._.20"
                        "xREP._.PREPROCESSED.parquet")

plate_map = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026_05_26"
                  "_JUNE-EXP_plate_map._.W-PERIODS.xlsx", engine='openpyxl')
)

timepoint_col = "Timepoint"
well_coll = "Well"
feature = "CalcResultI"
value = "mean"
meta_cols = ['group_label' , 'detected_period']
sampling_interval = 0.5

detrended_dataset = (
    pl.read_parquet(detrended_dataset)
    .filter(pl.col('feature') == "CalcResultI")
    .select(well_coll, timepoint_col, value)
    .join(plate_map, on=well_coll, how='left')
    .sort([timepoint_col, 'group_label'], descending=[False, False])
    .with_columns(pl.col(timepoint_col) + 6)
)

wells = detrended_dataset[well_coll].unique().to_list()

wave_res = calculate_wavelets(
    dataset=detrended_dataset,
    name_col='group_label',
    well_list=wells,
    well_col=well_coll,
    stat_val=value,
    timepoint_col=timepoint_col,
    sampling_interval=sampling_interval,
    period_range=(18,50),
    n_periods=250
)

Wells: 100%|██████████| 96/96 [00:01<00:00, 92.85it/s]


In [28]:
out_dir = Path("/home/derfelt/PartchLabFiles/Circadian "
               "Rhythms/Live Study/enVision/enVision_vi"
               "sual_results/2026-06-01")

out_file = out_dir / "06-01._.wavelet_analysis_visuals.pdf"

with PdfPages(out_file) as pdf:
    for fig in plot_wavelets(
        wavelet_results=wave_res,
        dataset=detrended_dataset,
        well_list=wells,
        well_col=well_coll,
        name_col='group_label',
        stat_val=value,
        timepoint_col=timepoint_col,
        tick_intervals=6
    ):
        pdf.savefig(fig)
        plt.close(fig)

well plots: 100%|██████████| 96/96 [00:22<00:00,  4.33it/s]


In [29]:
metrics = ['periods', 'amplitude', 'phase_hours', 'phase']
name_col = 'group_label'
timepoint_col = "Timepoint"

display(wave_res.ridge_df)

out_file = out_dir / "wavelet_ridge_results.html"
interactive_ridges(
    wavelet_res=wave_res,
    name_col=name_col,
    metrics=metrics,
    out_file=out_file
)

time,periods,phase,amplitude,power,frequencies,Well,group_label,phase_hours
f64,f64,f64,f64,f64,f64,str,str,f64
36.0,31.62249,5.045948,75.213674,13.848559,0.031623,"""F04""","""nan | nan""",25.395626
36.5,31.751004,5.156352,76.992186,14.570206,0.031495,"""F04""","""nan | nan""",26.056745
37.0,31.879518,5.261403,79.36618,15.545247,0.031368,"""F04""","""nan | nan""",26.695214
37.5,31.879518,5.352917,81.050292,16.211972,0.031368,"""F04""","""nan | nan""",27.159539
38.0,31.879518,5.449278,82.035203,16.608377,0.031368,"""F04""","""nan | nan""",27.648454
…,…,…,…,…,…,…,…,…
114.0,29.180723,1.807481,81.091503,3.144327,0.034269,"""J01""","""nan | nan""",8.394407
114.5,28.923695,1.954792,77.299705,2.831982,0.034574,"""J01""","""nan | nan""",8.99859
115.0,28.795181,2.08061,73.253863,2.531989,0.034728,"""J01""","""nan | nan""",9.535217
